# Uji refinement boundary di SongMap Debug

Buka dari root **SongMap**. Model worship + context yang sudah berhasil Anda jalankan tetap dipakai; tidak ada training, konversi, atau instalasi model baru.

Notebook mengompilasi **sumber Swift refiner yang dipakai aplikasi** dan membandingkan semua keputusan, anchor, skor kandidat, serta section akhir dengan Python. Ada 11 kasus: gerakan kedua arah, tidak berubah, bukti lemah/coverage hilang/gap/silence, dan JSON Who You Say I Am yang sudah disertakan. Pemeriksaan ini perlu lulus sebelum mencoba opsi Debug.

Untuk laporan contoh, hasil yang diharapkan adalah satu pergeseran **04:41.30 → 04:43.39**. Ini tidak menyatakan akurasi boundary umum atau perbaikan batas outro.


In [ ]:
from pathlib import Path
from datetime import datetime
import os, subprocess, sys

ROOT=Path.cwd().resolve()
assert (ROOT/'scripts/test_boundary_refinement.sh').is_file(), 'Buka notebook dari root SongMap'
receipt=ROOT/'SongMap/Resources/MLModels/SongFormerContext/songformer_section_context_manifest.json'
assert receipt.is_file(), 'Pasang model context dahulu lewat section_context_app_install.ipynb'
log=ROOT/'outputs.local'/('boundary-debug-check-'+datetime.now().strftime('%Y%m%d-%H%M%S-%f')+'.log')
log.parent.mkdir(parents=True,exist_ok=True)
command=['bash',str(ROOT/'scripts/test_boundary_refinement.sh')]
with log.open('w',encoding='utf-8') as handle:
    process=subprocess.Popen(command,cwd=ROOT,env={**os.environ,'BOUNDARY_REFINEMENT_PYTHON':sys.executable},
                             stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
    try:
        for line in process.stdout:
            print(line,end='');handle.write(line);handle.flush()
        status=process.wait()
    except KeyboardInterrupt:
        process.terminate();process.wait();raise
if status:
    raise RuntimeError(f'Pemeriksaan gagal ({status}); lihat {log}. Periksa sebelum lanjut build.')
print('Pemeriksaan Swift/Python selesai. Log:',log)


In [ ]:
project=ROOT/'SongMap.xcodeproj'
assert (project/'project.pbxproj').is_file(), 'Proyek Xcode belum ada; gunakan proyek SongMap yang sebelumnya berhasil dibuild'
subprocess.run(['open',str(project)],check=True)


Di Xcode:

1. Pilih iPhone dan konfigurasi **Debug**.
2. **Product → Scheme → Edit Scheme → Run → Arguments → Arguments Passed On Launch**.
3. Pakai satu argumen eksperimen: **`--songformer-worship-boundary-refinement`**. Nonaktifkan argumen eksperimen lama seperti `--songformer-worship-context`, `--songformer-worship-no-context`, `--songformer-legacy`, atau opsi `no-boundary-refinement`.
4. Build, jalankan, lalu **analisis ulang Who You Say I Am**. Hasil lama tidak diganti otomatis.
5. Dengarkan perpindahan bridge ke chorus sekitar **04:41–04:44**.

Log yang diharapkan: `WORSHIP ANALYSIS_START v10 ... boundary_refinement=debug-opt-in` lalu **`WORSHIP BOUNDARY_REFINEMENT_DONE debug-opt-in changed=1`** untuk laporan contoh. Nilai pada analisis baru bergantung pada bukti model/grid yang dihasilkan saat itu, sehingga jangan mengasumsikan jumlah pergeserannya pasti satu untuk semua lagu.

Pilihan Debug ini diingat saat membuka aplikasi dari Home Screen. Untuk mematikannya sambil mempertahankan model context, **ganti** argumen dengan **`--songformer-worship-no-boundary-refinement`** dan jalankan sekali. `--songformer-worship-context` juga mereset refinement ke nonaktif. Release tidak membaca preferensi ini.

Ekspor JSON lewat **Library → menu → Analysis reports → Share report**, lalu kirim ke sini. `boundaryRefinementEvidence` menyimpan batas asal, batas final, semua keputusan/abstention dan parameter. `contextEvidence` tetap menyimpan input asli model; `structureEvidence.assembledSections` berisi section akhir yang ditampilkan. Alat review Python dapat memeriksa ulang laporan baru dari batas asal, tanpa menggeser batas yang sama dua kali.

Refinement memakai probabilitas yang sudah dihitung dan grid aplikasi; tidak menjalankan encoder atau model context lagi. Label dan jumlah section tetap. Perubahan besar seperti outro yang seharusnya sekitar 05:15 tetapi masih 05:23 belum ditangani oleh eksperimen pergeseran lokal ini. Koreksi global semua lagu belum menjadi perilaku bawaan.
